In [1]:
homedir = '/home/annzhou/DRing/src/emp/datacentre/'
import random
import numpy as np

# from makec2s.ipynb
def genflowbytes():
    np.random.seed(0)
    
    mean_bytes = 100.0 * 1024
    shape = 1.05
    scale = mean_bytes * (shape - 1)/shape

    x = np.random.exponential(scale=1.0/shape)
    flowbytes = int(scale * np.exp(x))
    return flowbytes

def adjustbytesbymtu(flowbytes):
  mss = 1500
  return mss * ((flowbytes+mss-1)//mss)

large_flow_threshold = 10 * 1024 * 1024

In [2]:
stime = 192 # ms
nlinks = 2132 # 1066*2, uni-directional
nhosts = 2988
bw = 1342176000 # B per second
load_list = [10] # [20,50,80] # percentage
seed_list = [1,2,3,4,5]
topologytype = 2
nswitches = 80
os = 1
k = 64
nintervals = 8
topologyfile = 'evaltopologyfiles/dring_80_64.edgelist'
serverfile = 'evalserverfiles/dring_2988_80_64.sv'
npfile = 'evalnetpathfiles/netpath_dring_80_64_su3.np'

(current dir: ~/DRing/src/emp/datacentre/)
cp netpathfiles/netpath_su3_dring.txt evalnetpathfiles/netpath_dring_80_64_su3.np

In [ ]:
# # generate connection_matrices file (1)
# unv1bytes = 0
# unv1file = f'{homedir}rawtrafficfiles/unv1'
# maxinterval = 0
# with open(unv1file, 'r') as f:
#     lines = f.readlines()
#     for line in lines:
#         tokens = line.split(',')
#         # 0,32,31,10500
#         # interval,fromserver,toserver,bytes
#         unv1bytes += int(tokens[3])
#         maxinterval = max(maxinterval, int(tokens[0]))
# print(f'unv1bytes {unv1bytes}, maxinterval {maxinterval}, fullload {bw * stime * nlinks / 1000}, ratio {(bw * stime * nlinks / 1000) / unv1bytes}')

unv1bytes 162036861000, maxinterval 7, fullload 549411692544.0, ratio 3.3906587004545834


In [ ]:
# # generate connection_matrices file (2)
# random.seed(0)
# for load in load_list:
#     totalbytes = bw * stime * nlinks / 1000 * load / 100  # B
#     mult = totalbytes / unv1bytes
#     actualbytes = 0
#     cmfile = f'cmfiles/dring_load{load}.cm'
#     with open(cmfile, 'w') as fw:
#         with open(unv1file, 'r') as fr:
#             lines = fr.readlines()
#             iline = 0
#             while actualbytes < totalbytes:
#                 line = lines[iline]
#                 tokens = line.split(',')
#                 interval = int(tokens[0])
#                 fromserver = int(tokens[1])
#                 toserver = int(tokens[2])
#                 multbytes = int(tokens[3]) * mult

#                 if fromserver >= nhosts or toserver >= nhosts:
#                     iline += 1
#                     if iline >= len(lines):
#                         iline = 0
#                     continue

#                 # generate flows
#                 mybytes_sum = 0
#                 while mybytes_sum < multbytes:
#                     mybytes = genflowbytes()
#                     while mybytes<0 or mybytes>large_flow_threshold:
#                         mybytes = genflowbytes()
#                     mybytes = adjustbytesbymtu(mybytes)
#                     if mybytes_sum + mybytes > multbytes:
#                         mybytes = multbytes - mybytes_sum
#                         mybytes = adjustbytesbymtu(mybytes)
#                         break
#                     mybytes_sum += mybytes

#                     # generate random start time
#                     start_time_ms = random.uniform(0, stime//(maxinterval+1)) + interval * (stime//(maxinterval+1))

#                     fw.write(f'{fromserver},{toserver},{int(mybytes)},{start_time_ms:.4f}\n')
#                     actualbytes += int(mybytes)

#                 mybytes_sum += mybytes

#                 # generate random start time
#                 start_time_ms = random.uniform(0, stime//(maxinterval+1)) + interval * (stime//(maxinterval+1))

#                 fw.write(f'{fromserver},{toserver},{int(mybytes)},{start_time_ms:.4f}\n')
#                 actualbytes += int(mybytes)

#                 # print(f'multbytes {multbytes}, mybytes_sum {mybytes_sum}')
                
#                 iline += 1
#                 if iline >= len(lines):
#                     iline = 0

#     print(f'load {load}%, totalbytes {totalbytes}, unv1bytes {unv1bytes}, mult {mult}, actualbytes {actualbytes}')

load 10%, totalbytes 54941169254.4, unv1bytes 162036861000, mult 0.33906587004545835, actualbytes 54941212500


In [4]:
# generate pathweight file (1)
interval_stime = stime / nintervals
with open('dringsu3_generate_pwfiles.conf', 'w') as f:
    for load in load_list:
        cmfile = f'cmfiles/dring_load{load}.cm'
        for interval in range(nintervals):
            flowstart = interval_stime * interval
            flowend = interval_stime * (interval + 1)
            varfile = f'{homedir}rawpathweightfiles/pathtraffic_dring_{nhosts}_{nswitches}_{k}_su3_unv1_load{load}_interval{interval}.var'
            qvarfile = f'{homedir}rawpathweightfiles/pathweight_dring_{nhosts}_{nswitches}_{k}_su3_unv1_load{load}_interval{interval}.var'
            f.write(f"python3 {homedir}generate_pathweightfiles.py --graphfile {homedir}{topologyfile} --serverfile {homedir}{serverfile} --numsw {nswitches} --numserver {nhosts} --netpathfile {homedir}{npfile} --flowfile {cmfile} --flowstart {flowstart} --flowend {flowend} --numfaillink 0 --linkfailurefile none --varfile {varfile} --qvarfile {qvarfile}\n")

(current dir: ~/DRing/src/emp/datacentre/experiments/nsdi26fall/eval_main/unv1/)
python3 ../../../../pararun.py --conf dringsu3_generate_pwfiles.conf

In [5]:
# generate pathweight file (2)
intervaldict = {0:0,1:0,2:1,3:2,4:3,5:4,6:5,7:6} # to:from
with open('dringsu3_copy_pwfiles.conf', 'w') as f:
    for load in load_list:
        for interval in range(nintervals):
            fromfile = f'{homedir}rawpathweightfiles/pathweight_dring_{nhosts}_{nswitches}_{k}_su3_unv1_load{load}_interval{intervaldict[interval]}.var'
            tofile = f'{homedir}experiments/nsdi26fall/eval_main/unv1/pwfiles/pathweight_dring_su3_unv1_load{load}_interval{interval}.pw'
            f.write(f'cp {fromfile} {tofile}\n')

actually run the copy commands in datacentre/

In [6]:
# generate conf file
conffile = f'{homedir}experiments/nsdi26fall/eval_main/unv1/run_dringsu3.conf'
with open(conffile, 'w') as f:
    for seed in seed_list:
        for load in load_list:
            cmfile = f'experiments/nsdi26fall/eval_main/unv1/cmfiles/dring_load{load}.cm'
            pwfileprefix = f'experiments/nsdi26fall/eval_main/unv1/pwfiles/pathweight_dring_su3_unv1_load{load}_interval'
            outfile = f'experiments/nsdi26fall/eval_main/unv1/outfiles/dringsu3_load{load}.out'
            f.write(f"./eval -stime {stime} -seed {seed} -cmfile {cmfile} -topologytype {topologytype} -numswitches {nswitches} -numhosts {nhosts} -os {os} -ls_k {k} -npfile {npfile} -pwfileprefix {pwfileprefix} -numintervals {nintervals} -serverfile {serverfile} -topologyfile {topologyfile} > {outfile}\n")
            

python3 pararun.py --conf experiments/nsdi26fall/eval_main/unv1/run_dringsu3.conf